In [30]:
import pandas as pd
import numpy as np
import pickle
import ast
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

### 1. Load model

In [31]:
# Load Vietnamese SBERT model
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

   Embedding dimension: 768


### 2. Load embeddings

In [32]:
with open("recipes_embeddings_list.pkl", "rb") as f:
    recipes_embeddings_list = pickle.load(f)
print(f"Loaded {len(recipes_embeddings_list)} recipe embeddings")

Loaded 10335 recipe embeddings


### 3. Load dataset

In [33]:
all_recipes_df = pd.read_csv("../../data/all_recipes_final.csv")

print(f"Loaded dataset: {len(all_recipes_df)} recipes")
print(f"Columns: {all_recipes_df.columns.tolist()}")

Loaded dataset: 10335 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


### 3. Late Fusion Search Function

**Late Fusion Strategy:**
1. Tính similarity của query với **TẤT CẢ** các câu trong mỗi món
2. Lấy **trung bình** (average) similarity của tất cả câu → điểm số của món
3. Rank tất cả món theo điểm trung bình → Top K

In [34]:
def search_recipes_late_fusion(query, model, recipes_embeddings_list, all_recipes_df, top_k=10):
    """
    Search recipes using LATE FUSION strategy (Average Similarity)

    Late Fusion = Tính similarity với TẤT CẢ câu trong món → Average → Rank

    Args:
        query: User's search query (Vietnamese)
        model: SentenceTransformer model
        recipes_embeddings_list: List of embeddings per dish
        all_recipes_df: Recipe metadata dataframe
        top_k: Number of results to return

    Returns:
        DataFrame with top_k recipes and similarity scores
    """
    # 1. Encode query
    query_embedding = model.encode([query])
    query_embedding = query_embedding / np.linalg.norm(query_embedding)  # Normalize

    # 2. Calculate average similarity for EACH recipe
    recipe_scores = []

    for recipe_idx, dish_embeds in enumerate(recipes_embeddings_list):
        if len(dish_embeds) == 0:
            continue

        # Normalize dish embeddings
        dish_embeds_norm = dish_embeds / np.linalg.norm(dish_embeds, axis=1, keepdims=True)

        # Compute cosine similarity với TẤT CẢ câu
        similarities = np.dot(dish_embeds_norm, query_embedding.T).flatten()

        # LATE FUSION: Average similarity
        avg_similarity = np.mean(similarities)

        recipe_scores.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': float(avg_similarity),
            'max_similarity': float(np.max(similarities)),
            'min_similarity': float(np.min(similarities)),
            'num_sentences': len(similarities)
        })

    # 3. Sort by average similarity
    recipe_scores.sort(key=lambda x: x['avg_similarity'], reverse=True)
    top_recipes = recipe_scores[:top_k]

    # 4. Create results dataframe with FULL recipe info
    results = []
    for item in top_recipes:
        recipe_idx = item['recipe_idx']
        recipe = all_recipes_df.iloc[recipe_idx]

        results.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': item['avg_similarity'],
            'max_similarity': item['max_similarity'],
            'min_similarity': item['min_similarity'],
            'num_sentences': item['num_sentences'],
            'title': recipe['title'],
            'type_of_food': recipe['type_of_food'],
            'cook_time': recipe['cook_time'],
            'num_of_people': recipe['num_of_people'],
            'ingredients': recipe['ingredients'],
            'step': recipe['step'],
            'note': recipe['note'],
            'description': recipe['description'],
            'link': recipe['link']  # Thêm link
        })

    return pd.DataFrame(results)

### 4. Display results function

In [35]:
def parse_list_field(field_value):
    """
    Parse string / list field to Python list safely.
    """
    if pd.isna(field_value):
        return []

    if isinstance(field_value, list):
        return field_value

    if isinstance(field_value, str):
        try:
            parsed = ast.literal_eval(field_value)
            if isinstance(parsed, list):
                return parsed
            return []
        except (ValueError, SyntaxError):
            # fallback: split by comma
            return [s.strip() for s in field_value.split(",") if s.strip()]

    return []

In [36]:
import re

def display_results(results_df, query):
    """
    Display search results với TẤT CẢ thông tin món ăn

    Args:
        results_df: DataFrame from search_recipes_late_fusion
        query: Original query string
    """
    print(f"Query: '{query}'")
    print(f"Top {len(results_df)} Results:")

    for idx, row in results_df.iterrows():
        print(f"\n{'='*100}")
        print(f"{idx+1}. [{row['avg_similarity']:.4f}] {row['title']}")
        print(f"{'='*100}")

        # Basic info
        print(f"\nTHÔNG TIN CƠ BẢN:")
        print(f"   • Loại món: {row['type_of_food']}")
        print(f"   • Thời gian nấu: {row['cook_time']}")
        print(f"   • Số người ăn: {row['num_of_people']}")
        
        # Link
        if pd.notna(row['link']):
            print(f"   • Link: {row['link']}")

        # Similarity scores
        print(f"\nĐIỂM SIMILARITY:")
        print(f"   • Trung bình (AVG): {row['avg_similarity']:.4f}")
        print(f"   • Cao nhất (MAX): {row['max_similarity']:.4f}")
        print(f"   • Thấp nhất (MIN): {row['min_similarity']:.4f}")
        print(f"   • Số câu đánh giá: {row['num_sentences']}")

        # Description
        if pd.notna(row['description']):
            print(f"\nMÔ TẢ:")
            print(f"   {row['description']}")

        # Ingredients
        ingredients = parse_list_field(row['ingredients'])
        if ingredients:
            print(f"\nNGUYÊN LIỆU ({len(ingredients)} món):")
            for i, ing in enumerate(ingredients, 1):
                print(f"   {i}. {ing}")

        # Steps
        steps = parse_list_field(row['step'])
        if steps:
            # 1. Gộp tất cả step thành 1 chuỗi
            steps_text = " ".join(step.strip() for step in steps)

            # 2. Format: gặp "Bước X:" thì xuống dòng
            steps_text = re.sub(r'(Bước\s+\d+:)', r'\n\1', steps_text).strip()

            print(f"\nCÁCH LÀM:")
            print(steps_text)

        # Notes
        notes = parse_list_field(row['note'])
        if notes:
            print(f"\nLƯU Ý:")
            for i, note in enumerate(notes, 1):
                print(f"   • {note}")

        print()

In [37]:
# Test Late Fusion
test_queries = [
    "Món ăn có thịt bò nấu nhanh",
]

# Run Late Fusion tests
for query in test_queries:
    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=5
    )

    # Display results
    display_results(results, query)

Query: 'Món ăn có thịt bò nấu nhanh'
Top 5 Results:

1. [0.6043] Bò hầm cà rốt

THÔNG TIN CƠ BẢN:
   • Loại món: Món chính
   • Thời gian nấu: 30phút
   • Số người ăn: 2
   • Link: https://vncooking.com/cong-thuc/bo-ham-ca-rot-14

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.6043
   • Cao nhất (MAX): 0.7382
   • Thấp nhất (MIN): 0.4551
   • Số câu đánh giá: 4

MÔ TẢ:
   Bò luôn là món thịt mà đa phần các gia đình Việt rất ưa chuộng, bò chứa hàm lượng dinh dưỡng siêu cao cộng với cà rốt nữa làm tăng thêm phần dinh dưỡng của món ăn. Cùng chuẩn bị nguyên liệu thực hiện món Bò hầm cà rốt này nhé.

NGUYÊN LIỆU (7 món):
   1. Thịt bò 150 gram
   2. Cà rốt 2 củ
   3. Nước mắm 1 muỗng cafe
   4. Gừng 1 củ
   5. Muối 1 muỗng
   6. Dầu ăn 2 muỗng
   7. Sả 1 cây

CÁCH LÀM:
Bước 1: Nguyên liệu rửa sạch. Cà rốt cạo vỏ thái thành những hình vuông nhỏ, vừa ăn. Thịt bò cũng vậy, thái thành từng miếng hình vuống nhỏ thôi cho không bị day nha. Các nguyên liệu khác bỏ vỏ đun trên 1 nồi nước nhỏ, chờ khi nướ

### 5. Interactive Search

In [38]:
# Interactive search với Late Fusion
print("Enter your query (type 'quit' to exit):\n")

while True:
    query = input("Query: ").strip()

    if query.lower() in ['quit', 'exit', 'q']:
        break

    if not query:
        continue

    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=10
    )

    # Display results
    display_results(results, query)

Enter your query (type 'quit' to exit):

Query: 'gà chiên nước mắm'
Top 10 Results:

1. [0.6578] Gà tẩm bột chiên giòn Aji-Quick giòn rụm bằng chảo sâu lòng

THÔNG TIN CƠ BẢN:
   • Loại món: Món chiên
   • Thời gian nấu: 60 phút
   • Số người ăn: 4 người
   • Link: https://www.dienmayxanh.com/vao-bep/cach-lam-ga-tam-bot-chien-gion-aji-quick-gion-rum-bang-chao-23287

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.6578
   • Cao nhất (MAX): 0.7297
   • Thấp nhất (MIN): 0.6023
   • Số câu đánh giá: 5

MÔ TẢ:
   Bạn đang tìm cách làm gà tẩm bột chiên giòn Aji-Quick tại nhà? Cùng Vào bếp thực hiện món chiên giòn rụm, vàng ươm này. Công thức đơn giản, đảm bảo thành công ngay lần đầu, cho bữa ăn thêm hấp dẫn!

NGUYÊN LIỆU (4 món):
   1. 1 kg Thịt gà (xương gà hoặc các phần khác)
   2. 1 gói Bột chiên gà giòn Aji-Quick
   3. 1/2 chén Dầu ăn (khoảng 120ml)
   4. 1 ít Gia vị thông dụng (bột ngọt/tiêu xay)

CÁCH LÀM:
Bước 1: Sơ chế thịt gà: Thịt gà mua về, bạn rửa sạch rồi chặt thành từng miếng vừa ăn.

In [39]:
results

,recipe_idx,avg_similarity,max_similarity,min_similarity,num_sentences,title,type_of_food,cook_time,num_of_people,ingredients,step,note,description,link
0,6847,0.657802,0.729731,0.602273,5,Gà tẩm bột chiên giòn Aji-Quick giòn rụm bằng ...,Món chiên,60 phút,4 người,"['1 kg Thịt gà (xương gà hoặc các phần khác)',...","['Bước 1: Sơ chế thịt gà: Thịt gà mua về, bạn ...",['Xem chi tiết: Cách chọn mua gà ngon và cách ...,Bạn đang tìm cách làm gà tẩm bột chiên giòn Aj...,https://www.dienmayxanh.com/vao-bep/cach-lam-g...
1,1242,0.647897,0.719444,0.557065,4,Gà chiên sốt cay da giòn,Món chính,45phút,4,"['Gà phi lê 500 gram', 'Bột bắp 200 gram', 'Tr...",['Bước 1: - Gà thái miếng ướp với 1 muỗng muối...,[],Những miếng gà chiên giòn được tưới thêm nước ...,https://vncooking.com/cong-thuc/ga-chien-sot-c...
2,1124,0.637656,0.699988,0.513015,4,Gà nướng muối ớt thơm cay,Món chính,60phút,4,"['Gà 1 con', 'Tỏi băm 1 muỗng', 'Ớt bột 1 muỗn...",['Bước 1: - Gà rửa sạch cũng với ít nước muối ...,[],Gà nướng muối ớt được ướp bằng muối ớt được nê...,https://vncooking.com/cong-thuc/ga-nuong-muoi-...
3,35,0.636889,0.686852,0.586271,7,Cánh gà chiên bơ tỏi giòn ngon đổi vị ngày Tết,Món Tết,50 phút,4-6 người,"['6 cánh gà (800 gr)', '50 gr bơ lạt', '1 - 2 ...","['Bước 1: Cánh gà ngâm nước muối loãng, rửa sạ...",['Để rút ngắn thời gian chiên thì có thể luộc ...,"Cánh gà giòn rôm rốp, bên trong thịt ngọt mềm,...",https://vnexpress.net/doi-song-cooking-canh-ga...
4,1297,0.635412,0.673419,0.601519,4,Thịt gà chiên giòn trên từng miếng thịt,Món chính,45phút,4,"['Ức gà 300 gram', 'Bột chiên giòn 50 gram', '...","['Bước 1: - Thịt gà mua về rửa sạch, loại bỏ ...",[],"Thịt gà là món ăn được yêu thích tại Việt Nam,...",https://vncooking.com/cong-thuc/thit-ga-chien-...
5,847,0.633912,0.664444,0.572157,4,"Chân vịt rút xương chiên giòn nhanh gọn, đơn giản",Món nhậu,30phút,2,"['chân vịt rút xương 500 gam', 'Gừng 1 củ', 'B...",['Bước 1: Sơ chế chân vịt Chân vịt rút xương m...,[],Chân vịt rút xương chiên giòn là một món ăn th...,https://vncooking.com/cong-thuc/cach-lam-chan-...
6,440,0.624934,0.770646,0.549037,5,Cánh gà chiên nước mắm ngon tuyệt vời,Món ngon hàng ngày,45 phút,4-5 người,"['650 - 700 gram cánh gà.', '2 thìa canh đường...","['Bước 1: Cánh gà làm sạch, ngâm với nước ấm p...",[],"Cánh gà vàng óng, da giòn, thịt mềm ngọt, hươn...",https://vnexpress.net/doi-song-cooking-canh-ga...
7,4642,0.621614,0.722586,0.571927,6,Gà nướng muối ớt bằng than thơm lừng với chảo ...,Món nướng,40 phút,1 con gà (1.2 - 1.5 kg),"['1 con Gà ta (1.2 - 1.5 kg)', '1 ít Hành tím/...",['Bước 1: Sơ chế gà nguyên con: Gà ta mua về l...,['Xem thêm: Cách chọn mua gà ngon và cách sơ c...,Bạn đang tìm cách làm gà nướng muối ớt bằng th...,https://www.dienmayxanh.com/vao-bep/cach-lam-g...
8,411,0.621149,0.714584,0.353702,6,Gà lăn bột chiên xù,Món ngon hàng ngày,NaN,2 người,"['300g phi lê gà', '2 quả trứng gà', '200g bột...","['Bước 1: Gà cắt miếng vừa ăn, ướp hạt nêm và ...",[],Những món giòn giòn luôn tạo cảm giác ngon miệ...,https://vnexpress.net/ga-lan-bot-chien-xu-4311...
9,4761,0.620037,0.746111,0.528788,8,Gà nướng mật ong nồi chiên không dầu tiện lợi ...,Món nướng,45 phút,1 con gà (khoảng 1.2kg),"['1 con Gà (khoảng 1.2 kg)', '1 ít Gia vị ướp ...",['Bước 1: Sơ chế và chuẩn bị gà nguyên con: Gà...,['Xem chi tiết: Cách chọn mua gà ngon và cách ...,Bạn đang tìm kiếm công thức gà nướng mật ong n...,https://www.dienmayxanh.com/vao-bep/cach-lam-g...
